**The regression-based Leung's experiments**

In [ ]:
def count_elements(arr):
    counts = {'1': 0, '-1': 0, '0': 0}
    for element in arr:
        if element == 1:
            counts['1'] += 1
        elif element == -1:
            counts['-1'] += 1
        elif element == 0:
            counts['0'] += 1
    return counts

arr = [1, -1, 0, 1, 1, -1, 0, 0, 1, -1]
element_counts = count_elements(arr)
print(element_counts)

**CASE I (Baseline:) Unadjusted HT/Haj estimator**

In [ ]:
import numpy as np, networkx as nx, math
from inference_module import *
from data_module import *

np.set_printoptions(threshold = np.inf) 


Y,X,D,A,A_norm,pscores0,IDs = assemble_data()

print('D:', D)
print('len(D):', len(D) )
print('D:', count_elements(D))

A = A.to_undirected()
bandwidths = range(4)
school_IDs = [24, 56, 22, 60, 58]

# summary statistics
# network_stats(A, IDs, school_IDs)

##### Treatment effect #####

print('Treatment effect')

# units eligible for treatment
pop = (D >= 0)
print('D:', D)




print('pop', pop)
print('len(pop)', len(pop))

# summary statistics
node_stats(Y[pop], D[pop])
treat_pop_size = Y[pop].size

print('Y:', Y)
print('len(Y):', len(Y))

print('X:', X)
print('len(X):', len(X))


# results
n = Y.size #3306个
Z = make_Zs(  Y,    D,          1-D,         0.5*np.ones(n),0.5*np.ones(n),pop)  #Here is the treated - untreated
Z1 = make_Zs( Y,    D,  np.zeros(n),         0.5*np.ones(n),0.5*np.ones(n),pop)  # here is only the treated
Z0 = -make_Zs(Y,    np.zeros(n),1-D,         0.5*np.ones(n),0.5*np.ones(n),pop) #here is only the untreated   #These data are all observed!
estimate_treat = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()]) # compute the estimation
SE_treat = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])  # compute the variance


# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))




print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


##here is the direct  treatment effect...


#回归调整的G = D (160*2=320维向量)，尝试两种方案：1）做回归调整； 
#1) 确定这些点的X值，利用gender+grade构成自身的协变量；grade大家都一样啊。。先不管,已经提取出来了
#确定\beta 回归： \beta = 待做！！！


##### Spillover effect #####

print('\n\nSpillover effect')

# restrict to units with a friend eligible for treatment
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
has_treated_friends = (np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))) > 0) # > 0 treated friends

# summary statistics
node_stats(Y[pop], has_treated_friends[pop])
spill_pop_size = Y[pop].size

# results

n = Y.size 
Z = make_Zs(Y,   has_treated_friends,1-has_treated_friends,  1-pscores0,pscores0,pop)
Z1 = make_Zs(Y,  has_treated_friends,np.zeros(n),            1-pscores0,pscores0,pop)
Z0 = -make_Zs(Y, np.zeros(n),1-has_treated_friends,          1-pscores0,pscores0,pop)

estimate_spill = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()])
SE_spill = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])

# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))



print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


# mu(0) results
SE_mu0 = np.array([network_SE(Z0, A, pop, 1, True, False, b) for b in bandwidths])



##### Print results #####

table = pd.DataFrame(np.vstack([np.hstack([estimate_treat, SE_treat]), np.hstack([estimate_spill, SE_spill])]).T)
table.index = ['$\hat\\tau(1,0)$', '$\hat\\mu(1)$', '$\hat\\mu(0)$'] + ['$b_n = ' + str(i) + '$' for i in bandwidths]
table.columns = ['Treatment', 'Spillover']
print('\n\n\\begin{table}[ht]')
print('\centering')
print('\caption{Causal Effect Estimates and Standard Errors (HT)}')
print('\\begin{threeparttable}')
print(table.to_latex(float_format = lambda x: '%.4f' % x, header=True, escape=False))
print('\\begin{tablenotes}[para,flushleft]')
print("\\footnotesize Columns display results for the treatment ($n={}$) and spillover ($n={}$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.".format(treat_pop_size,spill_pop_size))
print('\end{tablenotes}')
print('\end{threeparttable}')
print('\end{table}')

print('\n\n\hat\mu(0) SEs')
print(SE_mu0)










In [ ]:
import numpy as np, networkx as nx, math
from inference_module import *
from data_module import *

np.set_printoptions(threshold = np.inf) 


Y,X,D,A,A_norm,pscores0,IDs = assemble_data()

print('D:', D)
print('len(D):', len(D) )
print('D:', count_elements(D))

A = A.to_undirected()


bandwidths = range(4)
school_IDs = [24, 56, 22, 60, 58]

# summary statistics
# network_stats(A, IDs, school_IDs)

##### Treatment effect #####

print('Treatment effect')

# units eligible for treatment
pop = (D >= 0)
print('D:', D)




print('pop', pop)
print('len(pop)', len(pop))

# summary statistics
node_stats(Y[pop], D[pop])
treat_pop_size = Y[pop].size

print('Y:', Y)
print('len(Y):', len(Y))

print('X:', X)
print('len(X):', len(X))


# results
n = Y.size #3302个
Z = make_Zs_haj(  Y,    D,          1-D,         0.5*np.ones(n),0.5*np.ones(n),pop)  #Here is the treated - untreated
Z1 = make_Zs_haj( Y,    D,  np.zeros(n),         0.5*np.ones(n),0.5*np.ones(n),pop)  # here is only the treated
Z0 = -make_Zs_haj(Y,    np.zeros(n),1-D,         0.5*np.ones(n),0.5*np.ones(n),pop) #here is only the untreated   #These data are all observed!
estimate_treat = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()]) # compute the estimation
SE_treat = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])  # compute the variance


# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))




print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


##here is the direct  treatment effect...


#回归调整的G = D (160*2=320维向量)，尝试两种方案：1）做回归调整； 
#1) 确定这些点的X值，利用gender+grade构成自身的协变量；grade大家都一样啊。。先不管,已经提取出来了
#确定\beta 回归： \beta = 待做！！！


##### Spillover effect #####

print('\n\nSpillover effect')

# restrict to units with a friend eligible for treatment
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
has_treated_friends = (np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))) > 0) # > 0 treated friends

# summary statistics
node_stats(Y[pop], has_treated_friends[pop])
spill_pop_size = Y[pop].size

# results

n = Y.size 
Z = make_Zs_haj(Y,   has_treated_friends,1-has_treated_friends,  1-pscores0,pscores0,pop)
Z1 = make_Zs_haj(Y,  has_treated_friends,np.zeros(n),            1-pscores0,pscores0,pop)
Z0 = -make_Zs_haj(Y, np.zeros(n),1-has_treated_friends,          1-pscores0,pscores0,pop)





estimate_spill = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()])
SE_spill = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])

# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))



print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


# mu(0) results
SE_mu0 = np.array([network_SE(Z0, A, pop, 1, True, False, b) for b in bandwidths])



##### Print results #####

table = pd.DataFrame(np.vstack([np.hstack([estimate_treat, SE_treat]), np.hstack([estimate_spill, SE_spill])]).T)
table.index = ['$\hat\\tau(1,0)$', '$\hat\\mu(1)$', '$\hat\\mu(0)$'] + ['$b_n = ' + str(i) + '$' for i in bandwidths]
table.columns = ['Treatment', 'Spillover']
print('\n\n\\begin{table}[ht]')
print('\centering')
print('\caption{Causal Effect Estimates and Standard Errors (haj)}')
print('\\begin{threeparttable}')
print(table.to_latex(float_format = lambda x: '%.4f' % x, header=True, escape=False))
print('\\begin{tablenotes}[para,flushleft]')
print("\\footnotesize Columns display results for the treatment ($n={}$) and spillover ($n={}$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.".format(treat_pop_size,spill_pop_size))
print('\end{tablenotes}')
print('\end{threeparttable}')
print('\end{table}')

print('\n\n\hat\mu(0) SEs')
print(SE_mu0)










In [ ]:
print('X:', X)
print('X.shape:', X.shape)  # 3302个数据点

print('Y:', Y)
print('Y.shape:', Y.shape)  # 3302个数据点


# print('pop.shape:', pop.shape)   
# assign = pop.astype(int)
# print('assign:', assign)
# print('assign.shape:', assign.shape)






G = A_norm   #这是归一化的邻接特征矩阵


E = A_norm
E.data[:] = 1  #construct the adjcent matrix for preparation. 这是0-1邻接矩阵



A_ = (np.dot(E, E.T)>0).astype(int)   #这是A = E*E矩阵，符号与仿真实验相同























In [ ]:
print('X:', X)

**Our regression (HT-PNI)**

In [ ]:
from scipy.stats import binom, norm
from sklearn.preprocessing import scale
import numpy as np
from scipy.sparse.linalg import spsolve

print('Y1:', Y)

num_nb = np.sum(E, axis=1)

num_rep=10000

#direct treatment effect:

pop = (D >= 0)



def get_T(Z):
    return (E.dot(Z) > np.floor(num_nb/2)).astype(int)
#注意符号


def get_X(X, Z, G):
    return np.column_stack((Z, G.dot(Z), X, G.dot(X)))


# def get_orth_coef(X, G, num_rep=10000):
mom_mat = np.zeros((num_rep, 5))
for i in range(num_rep):
#         Z = np.random.binomial(1, r1, n)
    X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
#     T_vec = get_T(Z)
    w = D[pop]/(1-0.5) - (1 - D[pop])/0.5
    mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w)]

mom_mean = np.mean(mom_mat, axis=0)
get_orth_coef = mom_mean[1:]/mom_mean[0]   #ouput: parametors.

print('get_orth_coef:', get_orth_coef)


## add the intercept term!
# def get_X_intercept(X, Z, G):
#     return np.column_stack(( np.ones(len(X)), Z, G.dot(Z), X, G.dot(X)))


# mom_mat = np.zeros((num_rep, 6))
# for i in range(num_rep):
# #         Z = np.random.binomial(1, r1, n)
#     X_aug = get_X_intercept(X[pop], assign[pop], A_norm[pop][:, pop])
# #     T_vec = get_T(Z)
#     w = assign[pop]/(1-0.5) - (1 - assign[pop])/0.5
#     mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w), np.mean(X_aug[:, 4]*w)]

# mom_mean = np.mean(mom_mat, axis=0)
# get_orth_coef_intercept = mom_mean[1:]/mom_mean[0]

# print('get_orth_coef_intercept:', get_orth_coef_intercept)


X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
X_db_PNI = X_aug - np.dot(w.reshape(-1, 1), get_orth_coef.reshape(1, -1))
print('X_db_PNI', X_db_PNI)

print('Y2:', Y)


D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
A_ = (np.dot(A_norm[pop][:, pop], A_norm[pop][:, pop]).T>0).astype(int)
matrix_left = (D_2.T @ A_) @ D_2
# matrix_left =np.array(matrix_left,dtype='float')
matrix_right = (D_2.T@ A_) @ (Y[pop]* w - Z[pop].mean())
# matrix_right =np.array(matrix_right,dtype='float')
hbeta_1 = spsolve(matrix_left, matrix_right)
print('Y3:', Y)


print('hbeta_1:', hbeta_1)

####有问题！！


hbeta_1 = [1,1,1,1]

Y[pop] = Y[pop] - np.dot(X_db_PNI[:,1:3], hbeta_1[1:3]) 

# Y_adj = Y

print('Y4:', Y)


#########################################################################regression-based experiment:
# Y_adj = Y
# Y_adj[pop] = Y[pop] - np.dot(X_db_PNI,    get_orth_coef)
#使用Y_new[pop]替换掉之前的Y,得到新的点估计

#test and debug:
# Y,X,D,A,A_norm,pscores0,IDs = assemble_data()
# Y_adj = Y




n = Y.size #3306个
Z = make_Zs(  Y,    D,          1-D,         0.5*np.ones(n),0.5*np.ones(n),pop)  #Here is the treated - untreated
Z1 = make_Zs( Y,    D,  np.zeros(n),         0.5*np.ones(n),0.5*np.ones(n),pop)  # here is only the treated
Z0 = -make_Zs(Y,    np.zeros(n),1-D,         0.5*np.ones(n),0.5*np.ones(n),pop) #here is only the untreated   #These data are all observed!
estimate_treat = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()]) # compute the estimation
SE_treat = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])  # compute the variance


# X_db = X_db_PNI


# D_2 = scale(X_db * np.array(w).reshape(-1,1), axis=0, with_std=False)
# #     print('D_2:', D_2)
# hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A), (D - tau_unadj_ht)))
# hbeta_2 = np.linalg.solve(np.dot(np.dot(D_2.T, A_p), D_2), np.dot(np.dot(D_2.T, A_p), (D - tau_unadj_ht)))

# tau_adj_aug1 = np.mean((Y - np.dot(X_db, hbeta_1)) * w)
# tau_adj_aug2 = np.mean((Y - np.dot(X_db, hbeta_2)) * w)
    
# #     bias1 = np.mean((Y - np.dot(X_aug, hbeta_1)) * w) - tau_adj_aug1
# #     bias2 = np.mean((Y - np.dot(X_aug, hbeta_2)) * w) - tau_adj_aug2
# #     print('bias1:', bias1)
# #     print('bias2:', bias2)

# var_est_adj_aug1 = np.dot(np.dot((D - np.dot(D_2, hbeta_1) - tau_adj_aug1).T, A), (D - np.dot(D_2, hbeta_1) - tau_adj_aug1)) / n**2
# var_est_adj_aug2 = np.dot(np.dot((D - np.dot(D_2, hbeta_2) - tau_adj_aug2).T, A_p), (D - np.dot(D_2, hbeta_2) - tau_adj_aug2)) / n**2

# is_cover_adj_aug_1 = abs(tau_adj_aug1 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug1)
# is_cover_adj_aug_2 = abs(tau_adj_aug2 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug2)





# var_est_adj_aug1 = ((Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15).T@ A_ @ (Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15)) / len(D[pop])**2
# var_est_adj_aug1 = ((Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15).T@ A_ @ (Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15)) / len(D[pop])**2
print(np.sqrt(((Y[pop]*w - np.dot(X_db_PNI[:,1:3], hbeta_1[1:3])* w ).T@ A_ @ (Y[pop]*w - np.dot(X_db_PNI, hbeta_1)* w ))/ (len(Y[pop])**2)))



# print('var_est_adj_aug1:', var_est_adj_aug1)












# orth_coef_intercept = get_orth_coef_intercept(X,A_norm)


































In [ ]:
print(X_db_PNI)
print(w.shape)
print('w:', w)
print(np.dot(w.T, X_db_PNI))
print('hbeta1:', hbeta_1)
print('Y[pop]*w:', np.dot(Y[pop],w)/len(Y[pop]))
print('Y[pop]:', Y[pop])
print('np.dot(X_db_PNI[:,1:3], hbeta_1[1:3])* w:', np.dot(X_db_PNI, hbeta_1)* w)
print('X_db_PNI:', X_db_PNI)
print('hbeta_1:', hbeta_1)
hbeta_1 = [0.1,0.1,0.1,0.1]
print(np.sqrt(((Y[pop]*w - 0.5*np.dot(X_db_PNI, hbeta_1)* w ).T@ A_ @ (Y[pop]*w - 0.5*np.dot(X_db_PNI, hbeta_1)* w ))/ (len(Y[pop])**2)))


print((((Y[pop]*w - 0.15 ).T@ A_ @ (Y[pop]*w - 0.15))/ (len(Y[pop])**2)))

In [ ]:

##### Spillover effect #####

print('\n\nSpillover effect')
Y,X,D,A,A_norm,pscores0,IDs = assemble_data()

A = A.to_undirected()


# restrict to units with a friend eligible for treatment
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
has_treated_friends = (np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))) > 0) # > 0 treated friends

has_over_half_friends =  (np.squeeze(np.asarray(A_norm.dot(np.ones(A_norm.shape[0])))) > 8)
# print( A_norm.dot(np.ones(A_norm.shape[0])))



# def get_orth_coef(X, G, num_rep=10000):
mom_mat = np.zeros((num_rep, 5))

# print('assign:', assign)
# print('T_vec:', get_T(assign)[pop])

print('pscores0:', pscores0)


for i in range(num_rep):
#         Z = np.random.binomial(1, r1, n)
    X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
#     T_vec = get_T(assign)
    w = has_treated_friends[pop]/(1-pscores0)[pop] - (1 - has_treated_friends[pop])/np.array(pscores0)[pop]
    mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w)]

mom_mean = np.mean(mom_mat, axis=0)
get_orth_coef = mom_mean[1:]/mom_mean[0]   #ouput: parametors.

print('get_orth_coef:', get_orth_coef)




X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
X_db_PNI = X_aug - np.dot(w.reshape(-1, 1), get_orth_coef.reshape(1, -1))
print('X_db_PNI', X_db_PNI)


# Y_adj = Y
D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
A_ = (np.dot(A_norm[pop][:, pop], A_norm[pop][:, pop].T)>0).astype(int)
matrix_left = (D_2.T @ A_) @ D_2
# matrix_left =np.array(matrix_left,dtype='float')
matrix_right = (D_2.T@ A_) @ (Y[pop]* w - Z[pop].mean())
# matrix_right =np.array(matrix_right,dtype='float')
hbeta_1 = spsolve(matrix_left, matrix_right)

hbeta_1 = [1,1,1,1]

Y[pop] = Y[pop] - np.dot(X_db_PNI[:,1:3], hbeta_1[1:3]) 



# D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
# A_ = np.dot(A_norm[pop][:, pop], A_norm[pop][:, pop])
# hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A_), (Y[pop]* w - Z[pop].mean())) )
# Y_adj[pop] = Y[pop] - np.dot(X_db_PNI, hbeta_1) 

    
    
#########################################################################regression-based experiment:
# Y_adj = Y
# Y_adj[pop] = Y[pop] - np.dot(X_db_PNI,    get_orth_coef)









# summary statistics
node_stats(Y[pop], has_treated_friends[pop])
spill_pop_size = Y[pop].size

# results

n = Y.size 
Z = make_Zs(Y,   has_treated_friends,1-has_treated_friends,  1-pscores0,pscores0,pop)
Z1 = make_Zs(Y,  has_treated_friends,np.zeros(n),            1-pscores0,pscores0,pop)
Z0 = -make_Zs(Y, np.zeros(n),1-has_treated_friends,          1-pscores0,pscores0,pop)





estimate_spill = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()])
SE_spill = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])


print('Z1:', len(Z1))
print('Z0:', len(Z0))



print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


# mu(0) results
SE_mu0 = np.array([network_SE(Z0, A, pop, 1, True, False, b) for b in bandwidths])



##### Print results #####

table = pd.DataFrame(np.vstack([np.hstack([estimate_treat, SE_treat]), np.hstack([estimate_spill, SE_spill])]).T)
table.index = ['$\hat\\tau(1,0)$', '$\hat\\mu(1)$', '$\hat\\mu(0)$'] + ['$b_n = ' + str(i) + '$' for i in bandwidths]
table.columns = ['Treatment', 'Spillover']
print('\n\n\\begin{table}[ht]')
print('\centering')
print('\caption{Causal Effect Estimates and Standard Errors (HT)}')
print('\\begin{threeparttable}')
print(table.to_latex(float_format = lambda x: '%.4f' % x, header=True, escape=False))
print('\\begin{tablenotes}[para,flushleft]')
print("\\footnotesize Columns display results for the treatment ($n={}$) and spillover ($n={}$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.".format(treat_pop_size,spill_pop_size))
print('\end{tablenotes}')
print('\end{threeparttable}')
print('\end{table}')

print('\n\n\hat\mu(0) SEs')
print(SE_mu0)



var_est_adj_aug1 = ((Y[pop]*w - np.dot(D_2, hbeta_1) - 0.04).T@ A_ @ (Y[pop]*w - np.dot(D_2, hbeta_1) - 0.04)) / len(D[pop])**2




In [ ]:

print('X_db_PNI:', X_db_PNI)

print('w:', w)
print(np.dot(w.T, X_db_PNI))

print(np.dot(w.T, Y[pop])/len(Y[pop]))
print('has_over_half_friends:', has_over_half_friends)

print('has_treated_friends:', has_treated_friends)



In [ ]:
print(A_norm)
print(np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))))
print( A_norm.dot(np.ones(A_norm.shape[0])))
print( A_norm.dot(D==1))

In [ ]:
Y,X,D,A,A_norm,pscores0,IDs = assemble_data()
print(X_db_PNI)
print(w.shape)
print('w:', w)
print(np.dot(w.T, X_db_PNI))
print('hbeta1:', hbeta_1)
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
print('pop:', pop)




print('Y[pop]*w:', np.dot(Y[pop],w)/len(Y[pop]))
print('Y[pop]:', Y[pop])

print('var_est_adj_aug1:', var_est_adj_aug1)


In [ ]:
print(len(D[pop]))

D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)

hbeta_1 = [-0.3,0.3,-0.3,0.3]  #可使用MC搜索
print(np.sqrt(((Y[pop]*w - np.dot(X_db_PNI, hbeta_1)* w ).T@ A_ @ (Y[pop]*w - np.dot(X_db_PNI, hbeta_1)* w ))/ (len(Y[pop])**2)))


print(np.sqrt(((Y[pop]*w - 0.04 ).T@ A_ @ (Y[pop]*w - 0.04 ))/ (len(Y[pop])**2)))

print('X_db_PNI:', X_db_PNI)
hbeta_1 = [1,1,1,1]
print('hbeta_1:', hbeta_1)
print('np.dot(X_db_PNI, hbeta_1)* w :', np.dot(X_db_PNI, hbeta_1)* w )
print('Y[pop]*w:', Y[pop]*w)

print('D_2:', D_2)
print('Y[pop]*w:', Y[pop]*w)
print('np.dot(D_2, hbeta_1):', np.dot(D_2, hbeta_1))
print('Y:', Y)

In [ ]:
print(np.dot(X_db_PNI, hbeta_1).shape)


In [ ]:
print('A_:', A_)